# CRNN Baseline — Kaggle GPU Training

Trains the Devanagari CRNN baseline on Kaggle's free GPU (~30 hrs/week quota).

**Before running — in the right sidebar:**
1. **Session options → Accelerator → GPU T4 x2** (or P100)
2. **Session options → Internet → On** (needed for `git clone`; requires phone-verified account)
3. **Input → Add Input** → search your uploaded dataset (`devnagari-preprocessed`) → **+**

Then run cells top to bottom. See `docs/GPU_TRAINING.md` for details.

In [ ]:
# 1. Confirm GPU is on
import torch
assert torch.cuda.is_available(), "No GPU! Sidebar -> Session options -> Accelerator -> GPU"
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the code (branch: ml). Kaggle preinstalls torch/cv2/numpy/tqdm/sklearn.
!git clone -b ml https://github.com/Sanskriti-poudel/Devnagari_Handwriting_Recognition.git
%cd Devnagari_Handwriting_Recognition

In [ ]:
# 3. Point the data loader at the attached dataset (auto-detects the slug)
import glob, os
candidates = glob.glob("/kaggle/input/*/train")
assert candidates, "Dataset not attached! Sidebar -> Input -> Add Input -> your devnagari dataset"
os.environ["DEVNAGARI_DATA_ROOT"] = os.path.dirname(candidates[0])
print("DEVNAGARI_DATA_ROOT =", os.environ["DEVNAGARI_DATA_ROOT"])
print(os.listdir(os.environ["DEVNAGARI_DATA_ROOT"]))  # expect: ['train', 'test']

In [ ]:
# 4. (Optional) Scan for corrupt images — fast on Kaggle's local disk,
#    and training skips bad images on its own anyway.
!python Preprocessing/verify_dataset.py --out /kaggle/working/corrupt_images.txt || true

In [ ]:
# 5. Train (auto-detects CUDA; batch_size defaults to 64 on GPU,
#    override with CRNN_BATCH_SIZE if you hit out-of-memory)
!python models/crnn/train.py

In [ ]:
# 6. Copy outputs to /kaggle/working so they appear in the Output panel
!cp models/crnn/checkpoints/best_model.pth /kaggle/working/
!cp logs/crnn_training.csv /kaggle/working/
print("Done. Download best_model.pth + crnn_training.csv from the Output panel",
      "(right sidebar -> Output), or Save Version to persist them.")

## After training

- **Download** `best_model.pth` + `crnn_training.csv` from the Output panel, then upload
  `best_model.pth` to the team Drive (weights are git-ignored — Contract A in `plans/README.md`).
- `crnn_training.csv` gives the training curves for the report (Phase 3).
- Next: evaluation on the held-out test set (Accuracy / CER / WER).